# Colab GPU runner (premonition)
Runs a job script that Claude already published as a Kaggle notebook, on this Colab GPU, then saves the results to your Google Drive (`MyDrive/premonition-runs/`) where Claude can read them.

**Before the first run:** Runtime → Change runtime type → **T4 GPU**. In the 🔑 Secrets panel (left sidebar) add `KAGGLE_API_TOKEN` (your KGAT_ token) and `LEARNER_ANTHROPIC_API_KEY`, and switch on notebook access for both.

Then **Runtime → Run all** and keep this tab open.

In [ ]:
JOB = "thegreatergood/rlaif-main"   # Claude tells you what to put here
RUN = JOB.split("/")[-1]

In [ ]:
import os, subprocess, shutil, glob
from google.colab import userdata, drive
os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")
os.environ["LEARNER_ANTHROPIC_API_KEY"] = userdata.get("LEARNER_ANTHROPIC_API_KEY")
drive.mount("/content/drive")
subprocess.run("pip install -q kaggle", shell=True, check=True)
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# the job scripts write to /kaggle/working, so point that at Colab's disk
os.makedirs("/content/kaggle/working", exist_ok=True)
if not os.path.exists("/kaggle"):
    os.symlink("/content/kaggle", "/kaggle")
src = "/content/jobsrc"; shutil.rmtree(src, ignore_errors=True)
subprocess.run(["kaggle", "kernels", "pull", JOB, "-p", src], check=True)
script = [p for p in glob.glob(src + "/*.py")][0]
print("running", script)
!cd /kaggle/working && python -u {script} 2>&1 | tee /kaggle/working/colab_run.log

In [ ]:
dst = f"/content/drive/MyDrive/premonition-runs/{RUN}"
os.makedirs(dst, exist_ok=True)
shutil.make_archive(dst + "/out", "gztar", "/kaggle/working", "out") if os.path.isdir("/kaggle/working/out") else None
shutil.copy("/kaggle/working/colab_run.log", dst)
print("saved to", dst, os.listdir(dst))